# Create SQLite Database

## Purpose

This notebook combines the validated monthly CSV snapshots into one dataset and creates a reproducible SQLite database for SQL analysis.

The notebook:

- loads all monthly snapshot files
- combines them into one DataFrame
- saves the combined dataset as a CSV
- creates the `missed_refunds.db` SQLite database
- writes the data to the `missed_refunds` table
- verifies that the database was created successfully

The database file is excluded from GitHub because it can be regenerated by running this notebook.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

data_folder = Path("../data/raw")

csv_files = sorted(data_folder.glob("*.csv"))

print(f"Found {len(csv_files)} CSV files")

Found 18 CSV files


In [2]:
# Read every monthly CSV file
dataframes = [pd.read_csv(file) for file in csv_files]

# Combine them into one DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

print(f"Combined rows: {len(combined_df)}")
print(f"Combined columns: {len(combined_df.columns)}")

Combined rows: 2871
Combined columns: 13


In [3]:
# Save the combined dataset
output_file = "../data/combined_missed_refunds.csv"

combined_df.to_csv(output_file, index=False)

print(f"Saved to: {output_file}")

Saved to: ../data/combined_missed_refunds.csv


## Create the SQLite Database

The combined dataset is written to a SQLite database so it can be queried in the SQL business-analysis notebook.

In [4]:
database_path = Path("../sql/missed_refunds.db")

database_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

connection = sqlite3.connect(database_path)

print(f"Connected to: {database_path}")

Connected to: ..\sql\missed_refunds.db


In [5]:
combined_df.to_sql(
    name="missed_refunds",
    con=connection,
    if_exists="replace",
    index=False,
)

print("Created table: missed_refunds")

Created table: missed_refunds


## Verify the Database

The table and row count are checked to confirm that the complete dataset was written successfully.

In [6]:
verification_query = """
SELECT COUNT(*) AS total_rows
FROM missed_refunds;
"""

pd.read_sql_query(
    verification_query,
    connection,
)

,total_rows
0,2871


In [7]:
connection.close()

print("Database connection closed.")

Database connection closed.
